# 01 · Data profiling
**Where to Invest: Microsoft Growth Investment Analysis**

Goal: understand the 8 raw system exports **before changing anything**: row counts, missing values,
duplicates, and the date and number formats each system uses. The output is a list of issues that
`02_cleaning.ipynb` fixes.

Tool: Python (pandas).

> **Synthetic data, calibrated to Microsoft's published results.** Generated by `data/generate.py`: totals by
> segment and quarter are close to Microsoft's reported figures; every record underneath them is invented.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

# Works whether the notebook is run from the project root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
REPORTS = ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

## 1. Load every file as text
Everything is read as text (`dtype=str`) so pandas does not silently convert dates or numbers.
Two files need special settings: billing is separated by `;`, and the budget has 2 metadata lines on top.

In [2]:
READ_OPTIONS = {
    "product_hierarchy.csv": {},
    "regions.csv": {},
    "treasury_fx_rates.csv": {},
    "erp_billing_lines.csv": {"sep": ";"},
    "gl_cost_postings.csv": {},
    "datacenter_regions.csv": {},
    "crm_opportunities.csv": {},
    "fpa_budget.csv": {"skiprows": 2},
}

raw = {
    name: pd.read_csv(RAW / name, dtype=str, keep_default_na=False, **options)
    for name, options in READ_OPTIONS.items()
}

# The two metadata lines of the budget file
with open(RAW / "fpa_budget.csv", encoding="utf-8") as f:
    budget_metadata = [next(f).strip() for _ in range(2)]
budget_metadata

['FP&A Planning Export - Version comparison by product line',
 'Units: USD thousands (synthetic data)']

## 2. Overview: rows, columns and size

In [3]:
overview = pd.DataFrame([
    {
        "file": name,
        "rows": len(df),
        "columns": df.shape[1],
        "size_mb": round((RAW / name).stat().st_size / 1e6, 1),
    }
    for name, df in raw.items()
]).sort_values("rows", ascending=False)

overview.loc["total"] = ["TOTAL", overview.rows.sum(), "", overview.size_mb.sum().round(1)]
overview

,file,rows,columns,size_mb
3,erp_billing_lines.csv,306684,14,32.1
4,gl_cost_postings.csv,120121,16,16.2
6,crm_opportunities.csv,41612,18,9.1
5,datacenter_regions.csv,1122,13,0.1
2,treasury_fx_rates.csv,462,5,0.0
7,fpa_budget.csv,300,18,0.1
0,product_hierarchy.csv,42,9,0.0
1,regions.csv,39,7,0.0
total,TOTAL,470382,,57.6


## 3. Column profile: missing values, distinct values, examples
A blank cell (`""`) counts as missing.

In [4]:
def profile(df: pd.DataFrame) -> pd.DataFrame:
    """One row per column: missing count and %, distinct values, three example values."""
    rows = []
    for col in df.columns:
        values = df[col]
        missing = (values.str.strip() == "").sum()
        examples = values[values.str.strip() != ""].drop_duplicates().head(3).tolist()
        rows.append({
            "column": col,
            "missing": int(missing),
            "missing_pct": round(100 * missing / len(df), 2),
            "distinct": values.nunique(),
            "examples": " | ".join(examples),
        })
    return pd.DataFrame(rows)


column_profiles = {name: profile(df) for name, df in raw.items()}

# Only the columns that have missing values, across all files
missing_summary = pd.concat(
    [p.assign(file=name) for name, p in column_profiles.items()]
).query("missing > 0")[["file", "column", "missing", "missing_pct"]]
missing_summary

,file,column,missing,missing_pct
8,erp_billing_lines.csv,Plant,238525,77.78
13,erp_billing_lines.csv,Reference Document,304250,99.21
8,gl_cost_postings.csv,Cost Center,594,0.49
9,gl_cost_postings.csv,Cost Center Name,594,0.49
15,gl_cost_postings.csv,Reversed With,102039,84.95
16,crm_opportunities.csv,Actual Close Date,4924,11.83


Most blanks are expected: `Plant` is only filled for Azure, `Reference Document` only for credit memos,
`Reversed With` only for reversal pairs and `Actual Close Date` only for closed deals. The real gap is the
**missing cost centre** in the general ledger (id and name).

In [5]:
# Full profile of the two largest files (change the name to see any other file)
column_profiles["erp_billing_lines.csv"]

,column,missing,missing_pct,distinct,examples
0,Billing Type,0,0.00,2,F2 | G2
1,Billing Document,0,0.00,190217,90000000 | 90000001 | 90000002
2,Item,0,0.00,8,000010 | 000020 | 000030
3,Billing Date,0,0.00,692,31.01.2024 | 29.02.2024 | 11.02.2024
4,Sold-To Party,0,0.00,7550,0020000000 | 0020000026 | 0020000052
5,Sold-To Name,0,0.00,7550,METROWOOD MEDIA LTD | KESTRELCREST BIOTECH PVT...
6,Country,0,0.00,39,AE | IN | JP
7,Material,0,0.00,40,400390 | 400120 | 400140
8,Plant,238525,77.78,35,JPE | WEU | FRC
9,Billed Quantity,0,0.00,176891,4026 | 24907 | 5030


In [6]:
column_profiles["gl_cost_postings.csv"]

,column,missing,missing_pct,distinct,examples
0,Company Code,0,0.00,23,IN10 | JP10 | NL10
1,Fiscal Year,0,0.00,4,2024 | 2025 | 2026
2,Posting Period,0,0.00,12,7 | 8 | 9
3,Document Number,0,0.00,120121,1000000680 | 1000003755 | 1000005585
4,Document Type,0,0.00,4,KR | PR | SA
5,Posting Date,0,0.00,1004,01.01.2024 | 02.01.2024 | 03.01.2024
6,G/L Account,0,0.00,13,500200 | 610200 | 620100
7,G/L Account Text,0,0.00,13,Cost of Revenue - Services | Partner Incentive...
8,Cost Center,594,0.49,164,100037 | 100185 | 100259
9,Cost Center Name,594,0.49,164,M365C-COR-IN | M365C-SAM-JP | M365C-SAM-NL


## 4. Duplicates
Two checks: exact duplicate rows, and duplicates of the column(s) that should be unique (the key).

In [7]:
EXPECTED_KEYS = {
    "product_hierarchy.csv": ["Material", "Valid From"],
    "regions.csv": ["Country Key"],
    "treasury_fx_rates.csv": ["Exchange Rate Type", "To Currency", "Valid From"],
    "erp_billing_lines.csv": ["Billing Document", "Item"],
    "gl_cost_postings.csv": ["Company Code", "Document Number"],
    "datacenter_regions.csv": ["Region Code", "Month"],
    "crm_opportunities.csv": ["Opportunity ID"],
    "fpa_budget.csv": ["Version", "Fiscal Year", "Product Line", "Line Item"],
}

duplicates = pd.DataFrame([
    {
        "file": name,
        "key": " + ".join(EXPECTED_KEYS[name]),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_keys": int(df.duplicated(subset=EXPECTED_KEYS[name]).sum()),
    }
    for name, df in raw.items()
])
duplicates

,file,key,duplicate_rows,duplicate_keys
0,product_hierarchy.csv,Material + Valid From,0,0
1,regions.csv,Country Key,0,0
2,treasury_fx_rates.csv,Exchange Rate Type + To Currency + Valid From,0,0
3,erp_billing_lines.csv,Billing Document + Item,0,0
4,gl_cost_postings.csv,Company Code + Document Number,0,0
5,datacenter_regions.csv,Region Code + Month,0,0
6,crm_opportunities.csv,Opportunity ID,0,0
7,fpa_budget.csv,Version + Fiscal Year + Product Line + Line Item,0,0


## 5. Date formats
Each system writes dates differently. A regular expression per format counts which ones appear in each date column.

In [8]:
DATE_PATTERNS = {
    "YYYY-MM-DD (ISO)": r"^\d{4}-\d{2}-\d{2}$",
    "MM/DD/YYYY (US)": r"^\d{2}/\d{2}/\d{4}$",
    "DD.MM.YYYY (European)": r"^\d{2}\.\d{2}\.\d{4}$",
    "YYYYMMDD (compact)": r"^\d{8}$",
    "YYYY-MM (month)": r"^\d{4}-\d{2}$",
}
DATE_COLUMNS = {
    "product_hierarchy.csv": ["Valid From", "Valid To"],
    "treasury_fx_rates.csv": ["Valid From"],
    "erp_billing_lines.csv": ["Billing Date"],
    "gl_cost_postings.csv": ["Posting Date"],
    "datacenter_regions.csv": ["Month"],
    "crm_opportunities.csv": ["Created On", "Est. Close Date", "Actual Close Date"],
}


def detect_format(values: pd.Series, patterns: dict) -> str:
    """Name the pattern(s) that match the non-blank values, with counts."""
    values = values[values.str.strip() != ""]
    found = {label: int(values.str.match(rx).sum()) for label, rx in patterns.items()}
    found = {k: v for k, v in found.items() if v}
    unmatched = len(values) - sum(found.values())
    text = ", ".join(f"{k}: {v:,}" for k, v in found.items())
    return text + (f", UNRECOGNISED: {unmatched:,}" if unmatched else "")


date_formats = pd.DataFrame([
    {"file": f, "column": c, "formats_found": detect_format(raw[f][c], DATE_PATTERNS)}
    for f, cols in DATE_COLUMNS.items() for c in cols
])
date_formats

,file,column,formats_found
0,product_hierarchy.csv,Valid From,YYYYMMDD (compact): 42
1,product_hierarchy.csv,Valid To,YYYYMMDD (compact): 42
2,treasury_fx_rates.csv,Valid From,YYYYMMDD (compact): 462
3,erp_billing_lines.csv,Billing Date,"DD.MM.YYYY (European): 306,684"
4,gl_cost_postings.csv,Posting Date,"DD.MM.YYYY (European): 120,121"
5,datacenter_regions.csv,Month,"YYYY-MM (month): 1,122"
6,crm_opportunities.csv,Created On,"MM/DD/YYYY (US): 41,612"
7,crm_opportunities.csv,Est. Close Date,"MM/DD/YYYY (US): 41,612"
8,crm_opportunities.csv,Actual Close Date,"MM/DD/YYYY (US): 36,688"


## 6. Number formats
Amounts arrive as text in several styles.

In [9]:
NUMBER_PATTERNS = {
    "plain 1234.56": r"^-?\d+(\.\d+)?$",
    "US 1,234.56": r"^-?\d{1,3}(,\d{3})+(\.\d+)?$",
    "decimal comma 1.234,56": r"^-?\d{1,3}(\.\d{3})*,\d+$",
    "trailing minus 1,234.56-": r"^[\d,]+\.\d+-$",
}
NUMBER_COLUMNS = {
    "treasury_fx_rates.csv": ["Exchange Rate"],
    "erp_billing_lines.csv": ["Net Value", "Billed Quantity"],
    "gl_cost_postings.csv": ["Amount in LC"],
    "datacenter_regions.csv": ["Capex Datacenter Build (USD)", "Utilization %", "Availability %"],
    "crm_opportunities.csv": ["Est. Revenue", "Probability (%)"],
    "fpa_budget.csv": ["Jul"],
}

number_formats = pd.DataFrame([
    {"file": f, "column": c, "formats_found": detect_format(raw[f][c], NUMBER_PATTERNS)}
    for f, cols in NUMBER_COLUMNS.items() for c in cols
])
number_formats

,file,column,formats_found
0,treasury_fx_rates.csv,Exchange Rate,plain 1234.56: 462
1,erp_billing_lines.csv,Net Value,"decimal comma 1.234,56: 306,684"
2,erp_billing_lines.csv,Billed Quantity,"plain 1234.56: 306,684"
3,gl_cost_postings.csv,Amount in LC,"plain 1234.56: 99, US 1,234.56: 110,981, trail..."
4,datacenter_regions.csv,Capex Datacenter Build (USD),"plain 1234.56: 1,122"
5,datacenter_regions.csv,Utilization %,"plain 1234.56: 1,122"
6,datacenter_regions.csv,Availability %,"plain 1234.56: 1,122"
7,crm_opportunities.csv,Est. Revenue,"US 1,234.56: 41,612"
8,crm_opportunities.csv,Probability (%),"plain 1234.56: 41,612"
9,fpa_budget.csv,Jul,"US 1,234.56: 300"


In [10]:
# Percentages: are they 0-1 or 0-100?
dc = raw["datacenter_regions.csv"]
dc[["Utilization %", "Availability %"]].astype(float).describe().loc[["min", "max"]]

,Utilization %,Availability %
min,47.7,99.901
max,97.0,99.999


## 7. Can the files be joined?
Every key that one file uses should exist in the file it points to.

In [11]:
billing = raw["erp_billing_lines.csv"]
gl = raw["gl_cost_postings.csv"]
opps = raw["crm_opportunities.csv"]
regions = raw["regions.csv"]
products = raw["product_hierarchy.csv"]


def coverage(child: pd.Series, parent: pd.Series) -> float:
    """% of distinct non-blank child values found in parent."""
    child = set(child[child != ""])
    return round(100 * len(child & set(parent)) / len(child), 2)


# datacenter cost centres end with their cloud region code: AZR-DCO-US-EUS -> EUS
dc_cost_centres = gl.loc[gl["Cost Center Name"].str.contains("-DCO-"), "Cost Center Name"]

join_checks = pd.DataFrame([
    ("Billing material found in product hierarchy", coverage(billing["Material"], products["Material"])),
    ("Billing country (ISO code) found in regions", coverage(billing["Country"], regions["Country Key"])),
    ("Billing plant found in datacenter regions", coverage(billing["Plant"], dc["Region Code"])),
    ("Billing currency found in FX rates (USD has no row)", coverage(billing["Document Currency"], raw["treasury_fx_rates.csv"]["To Currency"])),
    ("GL profit centre found in product hierarchy (PC-CORP has none)", coverage(gl["Profit Center"], products["Profit Center"])),
    ("GL company code found in regions", coverage(gl["Company Code"], regions["Company Code"])),
    ("GL datacenter cost centre region found in datacenter regions", coverage(dc_cost_centres.str.split("-").str[-1], dc["Region Code"])),
    ("Opportunity customer found among billed customers", coverage(opps["Customer ID"], billing["Sold-To Party"])),
    ("Opportunity country (full name) found in regions 'Country Key'", coverage(opps["Country/Region"], regions["Country Key"])),
    ("Opportunity country (full name) found in regions 'Country Name'", coverage(opps["Country/Region"], regions["Country Name"])),
    ("Opportunity product line found in product hierarchy", coverage(opps["Product Line"], products["Product Line"])),
    ("Budget product line found in product hierarchy", coverage(raw["fpa_budget.csv"]["Product Line"], products["Product Line"])),
], columns=["check", "match_pct"])
join_checks

,check,match_pct
0,Billing material found in product hierarchy,100.00
1,Billing country (ISO code) found in regions,100.00
2,Billing plant found in datacenter regions,100.00
3,Billing currency found in FX rates (USD has no...,87.50
4,GL profit centre found in product hierarchy (P...,92.31
5,GL company code found in regions,100.00
6,GL datacenter cost centre region found in data...,100.00
7,Opportunity customer found among billed customers,50.92
8,Opportunity country (full name) found in regio...,0.00
9,Opportunity country (full name) found in regio...,100.00


Four results below 100% are expected, not errors:
- **USD** has no row in the FX file (rate = 1).
- **PC-CORP** (corporate G&A) has no product line.
- **Opportunities** include prospects that never bought, so only about half of the CRM customers appear in billing.
- The **budget** has `Total …` subtotal rows that are not product lines.

The real issue is the country: the CRM stores **country names** while every other system uses **ISO codes**,
so the country must be mapped through `regions`.

## 8. File-specific checks

In [12]:
budget = raw["fpa_budget.csv"]

specific = pd.DataFrame([
    ("Billing: credit memos (type G2)", int((billing["Billing Type"] == "G2").sum())),
    ("Billing: lines with negative value", int(billing["Net Value"].str.startswith("-").sum())),
    ("Billing: currencies used", billing["Document Currency"].nunique()),
    ("GL: lines with a trailing minus (credits)", int(gl["Amount in LC"].str.endswith("-").sum())),
    ("GL: lines that reverse another document", int((gl["Reversed With"] != "").sum())),
    ("GL: posting date later than its posting period", 0),      # filled below
    ("GL: lines without a cost centre", int((gl["Cost Center"] == "").sum())),
    ("Products: materials with more than one validity row (segment move)", int(products["Material"].duplicated().sum())),
    ("Budget: segment subtotal rows ('Total ...')", int(budget["Product Line"].str.startswith("Total ").sum())),
    ("Budget: month columns (wide format)", sum(c in budget.columns for c in
        ["Jul", "Aug", "Sep", "Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun"])),
    ("Opportunities: open deals (no actual close date)", int((opps["Actual Close Date"] == "").sum())),
], columns=["check", "count"])

# Late postings: the posting date falls after the last day of its fiscal period
period = gl["Posting Period"].astype(int)
cal_month = (period + 5) % 12 + 1
cal_year = gl["Fiscal Year"].astype(int) - (period <= 6)
period_end = pd.to_datetime(dict(year=cal_year, month=cal_month, day=1)) + pd.offsets.MonthEnd(0)
posting_date = pd.to_datetime(gl["Posting Date"], format="%d.%m.%Y")
specific.loc[specific.check.str.startswith("GL: posting date"), "count"] = int((posting_date > period_end).sum())
specific

,check,count
0,Billing: credit memos (type G2),2434
1,Billing: lines with negative value,2434
2,Billing: currencies used,8
3,GL: lines with a trailing minus (credits),9041
4,GL: lines that reverse another document,18082
5,GL: posting date later than its posting period,1091
6,GL: lines without a cost centre,594
7,Products: materials with more than one validit...,2
8,Budget: segment subtotal rows ('Total ...'),60
9,Budget: month columns (wide format),12


Billing lines per month show the billing pattern: spikes at quarter end, because enterprise agreements are
billed quarterly in arrears.

In [13]:
billing_month = pd.to_datetime(billing["Billing Date"], format="%d.%m.%Y").dt.to_period("M")
billing_month.value_counts().sort_index().rename("billing_lines").to_frame().T

Billing Date,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,...,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
billing_lines,7116,7242,11767,7324,7321,11884,7308,7352,11993,7361,7386,12137,7448,7475,12309,...,7622,7674,12668,7710,7748,12893,7830,7888,13133,8015,8088,13519,8299,8407,14203


## 9. Issue list for the cleaning notebook

In [14]:
issues = pd.DataFrame([
    ("All", "Four date formats: MM/DD/YYYY, DD.MM.YYYY, YYYYMMDD and YYYY-MM", "Parse each column with its explicit format"),
    ("erp_billing_lines", "';' separator and decimal comma (1.234,56)", "Read with sep=';'; convert to float"),
    ("gl_cost_postings", "Trailing minus (1,234.56-) and thousand separators", "Move '-' to the front; convert to float"),
    ("crm_opportunities", "Amounts stored as text with ','", "Strip ','; convert to float"),
    ("crm_opportunities", "Country stored as a full name, not an ISO code", "Map to country_key through regions"),
    ("erp_billing_lines", "Credit memos booked months after the invoice", "Link to the original invoice via Reference Document"),
    ("gl_cost_postings", "Month-end accruals reversed in the next period", "Keep both lines (they net out)"),
    ("gl_cost_postings", "Wrong postings reversed in the same period", "Remove both lines of each pair; keep the corrected posting"),
    ("gl_cost_postings", "Late postings into closed periods", "Use fiscal period, not posting date; flag them"),
    ("gl_cost_postings", "Blank cost centre", "Keep; use Profit Center for product line"),
    ("product_hierarchy", "Windows Commercial changes segment on 1 Jul 2025", "Add restated segment for like-for-like"),
    ("fpa_budget", "2 metadata lines, wide months, subtotal rows, USD thousands", "Skip lines, drop subtotals, unpivot, x1000"),
    ("erp_billing_lines / gl_cost_postings / crm_opportunities", "8 currencies; USD has no FX row", "Convert with monthly average rates; add USD = 1"),
    ("datacenter_regions", "Percentages stored as 0-100", "Divide by 100"),
], columns=["file", "issue", "planned_fix"])

issues.to_csv(REPORTS / "profiling_issues.csv", index=False)
issues

,file,issue,planned_fix
0,All,"Four date formats: MM/DD/YYYY, DD.MM.YYYY, YYY...",Parse each column with its explicit format
1,erp_billing_lines,"';' separator and decimal comma (1.234,56)",Read with sep=';'; convert to float
2,gl_cost_postings,"Trailing minus (1,234.56-) and thousand separa...",Move '-' to the front; convert to float
3,crm_opportunities,"Amounts stored as text with ','","Strip ','; convert to float"
4,crm_opportunities,"Country stored as a full name, not an ISO code",Map to country_key through regions
5,erp_billing_lines,Credit memos booked months after the invoice,Link to the original invoice via Reference Doc...
6,gl_cost_postings,Month-end accruals reversed in the next period,Keep both lines (they net out)
7,gl_cost_postings,Wrong postings reversed in the same period,Remove both lines of each pair; keep the corre...
8,gl_cost_postings,Late postings into closed periods,"Use fiscal period, not posting date; flag them"
9,gl_cost_postings,Blank cost centre,Keep; use Profit Center for product line


**Next:** `02_cleaning.ipynb` applies each fix and logs every rule with the rows it touches.